In [ ]:
# include the parameters file and emg_samples file over here
from google.colab import files

# This will prompt you to select the 'int_x.txt' file from your computer
uploaded = files.upload()

Saving int_x.txt to int_x.txt
Saving mp_5_a_pca_ori_model_wt_bias.txt to mp_5_a_pca_ori_model_wt_bias.txt


In [ ]:
# maximum and minimum value of the individual emg data in entire file
import re

with open("int_x.txt", "r") as f:
    text = f.read()

numbers = [int(x) for x in re.findall(r"-?\d+", text)]

print("Highest value:", max(numbers))
print("Lowest value:", min(numbers))

Highest value: 1545
Lowest value: -758


In [ ]:
# maximum and minimum value of the individual emg data in specified sample
import re
import numpy as np

SAMPLE_INDEX = 14

with open("int_x.txt", "r") as f:
    text = f.read()

numbers = [int(x) for x in re.findall(r"-?\d+", text)]

# Each sample = 1019 time positions × 6 channels
SAMPLE_SIZE = 1019 * 6

start = SAMPLE_INDEX * SAMPLE_SIZE
end = start + SAMPLE_SIZE

sample = numbers[start:end]

if len(sample) != SAMPLE_SIZE:
    raise ValueError("Specified sample does not contain 1019 × 6 values.")

print("Sample:", SAMPLE_INDEX)
print("Highest value:", max(sample))
print("Lowest value:", min(sample))

Sample: 14
Highest value: 40
Lowest value: -106


In [ ]:
import re
import ast
import numpy as np
from pathlib import Path

In [1]:
import re
import ast
import numpy as np

# SAMPLE OUTPUT OF THIS BLOCK OF CODE
# First two rows of selected sample:
# [[-52 -25  40   8  19  -5]
#  [-56 -22  32   6  16  -4]]

# ========================================
# SOFTWARE NEURAL NETWORK REFERENCE
# ========================================
# Number of samples : 150
# Selected sample   : 14
# Input shape       : (1019, 6)

# Weight shapes:
# conv1d_8 : (2, 6, 32)  bias: (32,)
# conv1d_9 : (2, 32, 64)  bias: (64,)
# dense_8  : (64, 64)  bias: (64,)
# dense_9  : (64, 6)  bias: (6,)

# Conv1 output
#   Shape       : (1018, 32)
#   First 10    : [-6486   610   625 -3292  1824   365  5418  4415 -2030 -2641]
#   Minimum     : -12674
#   Maximum     : 9348

# ReLU1 output
#   Shape       : (1018, 32)
#   First 10    : [   0  610  625    0 1824  365 5418 4415    0    0]
#   Minimum     : 0
#   Maximum     : 9348

# MaxPool output
#   Shape       : (509, 32)
#   First 10    : [   0 1395  712    0 1824  365 5418 4616    0    0]
#   Minimum     : 0
#   Maximum     : 9348

# Conv2 output
#   Shape       : (508, 64)
#   First 10    : [ 349414  312931  358436  290148  355084  354371 -670406 -624260  418866
#  -175733]
#   Minimum     : -2494441
#   Maximum     : 1892769

# ReLU2 output
#   Shape       : (508, 64)
#   First 10    : [349414 312931 358436 290148 355084 354371      0      0 418866      0]
#   Minimum     : 0
#   Maximum     : 1892769

# Global Average Pool output
#   Shape       : (64,)
#   First 10    : [2.35378026e+05 7.46624016e+04 3.17666858e+05 3.59506614e+04
#  3.96051577e+05 5.27654075e+05 0.00000000e+00 1.85102362e+02
#  2.32878543e+04 2.13197303e+04]
#   Minimum     : 0.0
#   Maximum     : 1172869.037401575

# Dense1 output
#   Shape       : (64,)
#   First 10    : [-4.39536957e+07 -8.52774785e+07  3.29364204e+07  7.25092192e+07
#   5.49777556e+07  3.52027132e+07 -9.15833824e+07  1.15135134e+08
#  -8.86982717e+07  9.54605796e+07]
#   Minimum     : -205540349.06889763
#   Maximum     : 210839570.77952757

# ReLU after Dense1
#   Shape       : (64,)
#   First 10    : [0.00000000e+00 0.00000000e+00 3.29364204e+07 7.25092192e+07
#  5.49777556e+07 3.52027132e+07 0.00000000e+00 1.15135134e+08
#  0.00000000e+00 9.54605796e+07]
#   Minimum     : 0.0
#   Maximum     : 210839570.77952757

# Final logits
#   Shape       : (6,)
#   First 10    : [ 4.40615581e+10  1.32191613e+10  8.71313413e+09 -2.81967190e+10
#  -3.01071306e+10 -2.95761688e+09]
#   Minimum     : -30107130603.200783
#   Maximum     : 44061558055.484245

# Final logits:
# [ 4.40615581e+10  1.32191613e+10  8.71313413e+09 -2.81967190e+10
#  -3.01071306e+10 -2.95761688e+09]

# Softmax probabilities:
# [1. 0. 0. 0. 0. 0.]

# Predicted class: 0

# Full intermediate values saved to:
# intermediate_sample_14.txt

# ============================================================
# SETTINGS
# ============================================================

WEIGHT_FILE = "mp_5_a_pca_ori_model_wt_bias.txt"
SAMPLE_FILE = "int_x.txt"

# Which EMG sample to run.
# There are 150 samples, so valid values are 0 to 149.
SAMPLE_INDEX = 14

# Full intermediate arrays are written to this file.
OUTPUT_FILE = f"intermediate_sample_{SAMPLE_INDEX}.txt"


# ============================================================
# READ EMG SAMPLES
# ============================================================

def load_samples(filename):
    """
    The sample file is formatted approximately as:

    [[[ -49 -6 -2 34 18 2]
      [ -51 -11 2 26 14 1]
      ...
    ]

    Each sample has shape 1019 x 6.
    """

    text = Path(filename).read_text()

    # Extract every integer from the file.
    numbers = [int(x) for x in re.findall(r"-?\d+", text)]

    expected = 150 * 1019 * 6

    if len(numbers) != expected:
        raise ValueError(
            f"Expected {expected} integers in sample file, "
            f"but found {len(numbers)}."
        )

    return np.array(numbers, dtype=np.int64).reshape(150, 1019, 6)


# ============================================================
# READ WEIGHTS AND BIASES
# ============================================================

def load_weights(filename):
    """
    Reads the four layers from the supplied weight text file.

    Expected layers:

        conv1d_8 : (2, 6, 32)
        conv1d_9 : (2, 32, 64)
        dense_8  : (64, 64)
        dense_9  : (64, 6)
    """

    text = Path(filename).read_text()

    layer_names = [
        "conv1d_8",
        "conv1d_9",
        "dense_8",
        "dense_9"
    ]

    result = {}

    for i, name in enumerate(layer_names):

        start = text.index(f"Layer: {name}")

        if i + 1 < len(layer_names):
            end = text.index(f"Layer: {layer_names[i + 1]}")
        else:
            end = len(text)

        section = text[start:end]

        weight_text = section.split(
            "Quantized Weights:", 1
        )[1].split(
            "Quantized Biases:", 1
        )[0].strip()

        bias_text = section.split(
            "Quantized Biases:", 1
        )[1].strip()

        # Remove the dashed separator after the bias.
        bias_text = bias_text.split("-" * 10)[0].strip()

        weights = np.array(
            ast.literal_eval(weight_text),
            dtype=np.int64
        )

        biases = np.array(
            ast.literal_eval(bias_text),
            dtype=np.int64
        )

        result[name] = (weights, biases)

    return result


# ============================================================
# CONVOLUTION
# ============================================================

def conv1d(x, weights, bias):
    """
    Simple valid 1-D convolution/cross-correlation.

    x:
        (time, input_channels)

    weights:
        (kernel_size, input_channels, output_channels)

    output:
        (time-kernel_size+1, output_channels)
    """

    time_length, input_channels = x.shape
    kernel_size, weight_channels, output_channels = weights.shape

    if input_channels != weight_channels:
        raise ValueError("Input channel count does not match weights.")

    output_length = time_length - kernel_size + 1

    output = np.zeros(
        (output_length, output_channels),
        dtype=np.int64
    )

    for t in range(output_length):
      for k in range(kernel_size):
          for c in range(input_channels):
              output[t] += x[t + k, c] * weights[k, c]
      output[t] += bias
    return output

# ============================================================
# PRINT / SAVE INTERMEDIATE VALUES
# ============================================================

def write_array(f, name, array):
    f.write("\n")
    f.write("=" * 70 + "\n")
    f.write(f"{name}\n")
    f.write(f"Shape: {array.shape}\n")
    f.write("=" * 70 + "\n")
    f.write(np.array2string(
        array,
        separator=", ",
        threshold=np.inf,
        max_line_width=200
    ))
    f.write("\n")


def show_summary(name, array):
    flat = array.reshape(-1)

    print(f"\n{name}")
    print(f"  Shape       : {array.shape}")
    print(f"  First 10    : {flat[:10]}")
    print(f"  Minimum     : {flat.min()}")
    print(f"  Maximum     : {flat.max()}")


# ============================================================
# MAIN
# ============================================================

samples = load_samples(SAMPLE_FILE)
weights = load_weights(WEIGHT_FILE)

if not 0 <= SAMPLE_INDEX < len(samples):
    raise ValueError(
        f"SAMPLE_INDEX must be between 0 and {len(samples)-1}"
    )

x = samples[SAMPLE_INDEX]

print("\nFirst two rows of selected sample:")
print(x[:2])

# Get weights and biases.
w1, b1 = weights["conv1d_8"]
w2, b2 = weights["conv1d_9"]
w3, b3 = weights["dense_8"]
w4, b4 = weights["dense_9"]

print("\n========================================")
print("SOFTWARE NEURAL NETWORK REFERENCE")
print("========================================")

print(f"Number of samples : {len(samples)}")
print(f"Selected sample   : {SAMPLE_INDEX}")
print(f"Input shape       : {x.shape}")

print("\nWeight shapes:")
print("conv1d_8 :", w1.shape, " bias:", b1.shape)
print("conv1d_9 :", w2.shape, " bias:", b2.shape)
print("dense_8  :", w3.shape, " bias:", b3.shape)
print("dense_9  :", w4.shape, " bias:", b4.shape)


# ------------------------------------------------------------
# LAYER 1
# ------------------------------------------------------------

conv1 = conv1d(x, w1, b1)
relu1 = np.maximum(conv1, 0)

show_summary("Conv1 output", conv1)
show_summary("ReLU1 output", relu1)


# ------------------------------------------------------------
# MAX POOLING
# ------------------------------------------------------------

# Pool size = 2, stride = 2.
# 1018 becomes 509.

pool1 = relu1.reshape(
    relu1.shape[0] // 2,
    2,
    relu1.shape[1]
).max(axis=1)

show_summary("MaxPool output", pool1)


# ------------------------------------------------------------
# LAYER 2
# ------------------------------------------------------------

conv2 = conv1d(pool1, w2, b2)
relu2 = np.maximum(conv2, 0)

show_summary("Conv2 output", conv2)
show_summary("ReLU2 output", relu2)


# ------------------------------------------------------------
# GLOBAL AVERAGE POOLING
# ------------------------------------------------------------

gap = np.mean(relu2, axis=0)

show_summary("Global Average Pool output", gap)


# ------------------------------------------------------------
# DENSE 1
# ------------------------------------------------------------

dense1 = gap @ w3 + b3
relu3 = np.maximum(dense1, 0)

show_summary("Dense1 output", dense1)
show_summary("ReLU after Dense1", relu3)


# ------------------------------------------------------------
# DENSE 2 / CLASSIFIER
# ------------------------------------------------------------

logits = dense1 @ w4 + b4

# Softmax is only used to get probabilities.
# Prediction itself is simply argmax(logits).
exp_logits = np.exp(logits - np.max(logits))
probabilities = exp_logits / np.sum(exp_logits)

prediction = int(np.argmax(logits))

show_summary("Final logits", logits)

print("\nFinal logits:")
print(logits)

print("\nSoftmax probabilities:")
print(probabilities)

print("\nPredicted class:", prediction)


# ============================================================
# SAVE ALL INTERMEDIATE VALUES
# ============================================================

with open(OUTPUT_FILE, "w") as f:

    f.write("SOFTWARE INTERMEDIATE VALUES\n")
    f.write(f"Sample index: {SAMPLE_INDEX}\n")

    write_array(f, "INPUT", x)

    write_array(f, "CONV1 OUTPUT", conv1)
    write_array(f, "RELU1 OUTPUT", relu1)

    write_array(f, "MAXPOOL OUTPUT", pool1)

    write_array(f, "CONV2 OUTPUT", conv2)
    write_array(f, "RELU2 OUTPUT", relu2)

    write_array(f, "GLOBAL AVERAGE POOL OUTPUT", gap)

    write_array(f, "DENSE1 OUTPUT", dense1)
    write_array(f, "RELU AFTER DENSE1", relu3)

    write_array(f, "FINAL LOGITS", logits)
    write_array(f, "SOFTMAX PROBABILITIES", probabilities)

    f.write("\n")
    f.write("=" * 70 + "\n")
    f.write(f"PREDICTED CLASS: {prediction}\n")
    f.write("=" * 70 + "\n")

print(f"\nFull intermediate values saved to:")
print(OUTPUT_FILE)

NameError: name 'Path' is not defined

In [ ]:
import re
import ast
import numpy as np

# I dont fully remember what this block is doing. The hardware had 8-bit quantisation and that was causing issues because baseline model didnt had so. Hence I guess this did the same thing
# as the above block but with 8 bit quantisation applied to each layers ouput


# ============================================================
# SETTINGS
# ============================================================

WEIGHT_FILE = "mp_5_a_pca_ori_model_wt_bias.txt"
SAMPLE_FILE = "int_x.txt"

# Which EMG sample to run.
# There are 150 samples, so valid values are 0 to 149.
SAMPLE_INDEX = 36

# Full intermediate arrays are written to this file.
OUTPUT_FILE = f"intermediate_sample_{SAMPLE_INDEX}.txt"


# ============================================================
# READ EMG SAMPLES
# ============================================================

def load_samples(filename):
    """
    The sample file is formatted approximately as:

    [[[ -49 -6 -2 34 18 2]
      [ -51 -11 2 26 14 1]
      ...
    ]

    Each sample has shape 1019 x 6.
    """

    text = Path(filename).read_text()

    # Extract every integer from the file.
    numbers = [int(x) for x in re.findall(r"-?\d+", text)]

    expected = 150 * 1019 * 6

    if len(numbers) != expected:
        raise ValueError(
            f"Expected {expected} integers in sample file, "
            f"but found {len(numbers)}."
        )

    return np.array(numbers, dtype=np.int64).reshape(150, 1019, 6)


# ============================================================
# READ WEIGHTS AND BIASES
# ============================================================

def load_weights(filename):
    """
    Reads the four layers from the supplied weight text file.

    Expected layers:

        conv1d_8 : (2, 6, 32)
        conv1d_9 : (2, 32, 64)
        dense_8  : (64, 64)
        dense_9  : (64, 6)
    """

    text = Path(filename).read_text()

    layer_names = [
        "conv1d_8",
        "conv1d_9",
        "dense_8",
        "dense_9"
    ]

    result = {}

    for i, name in enumerate(layer_names):

        start = text.index(f"Layer: {name}")

        if i + 1 < len(layer_names):
            end = text.index(f"Layer: {layer_names[i + 1]}")
        else:
            end = len(text)

        section = text[start:end]

        weight_text = section.split(
            "Quantized Weights:", 1
        )[1].split(
            "Quantized Biases:", 1
        )[0].strip()

        bias_text = section.split(
            "Quantized Biases:", 1
        )[1].strip()

        # Remove the dashed separator after the bias.
        bias_text = bias_text.split("-" * 10)[0].strip()

        weights = np.array(
            ast.literal_eval(weight_text),
            dtype=np.int64
        )

        biases = np.array(
            ast.literal_eval(bias_text),
            dtype=np.int64
        )

        result[name] = (weights, biases)

    return result


# ============================================================
# CONVOLUTION
# ============================================================

def conv1d(x, weights, bias):
    """
    Simple valid 1-D convolution/cross-correlation.

    x:
        (time, input_channels)

    weights:
        (kernel_size, input_channels, output_channels)

    output:
        (time-kernel_size+1, output_channels)
    """

    time_length, input_channels = x.shape
    kernel_size, weight_channels, output_channels = weights.shape

    if input_channels != weight_channels:
        raise ValueError("Input channel count does not match weights.")

    output_length = time_length - kernel_size + 1

    output = np.zeros(
        (output_length, output_channels),
        dtype=np.int64
    )

    for t in range(output_length):
      for k in range(kernel_size):
          for c in range(input_channels):
              output[t] += x[t + k, c] * weights[k, c]
      output[t] += bias
    return output

# ============================================================
# PRINT / SAVE INTERMEDIATE VALUES
# ============================================================

def write_array(f, name, array):
    f.write("\n")
    f.write("=" * 70 + "\n")
    f.write(f"{name}\n")
    f.write(f"Shape: {array.shape}\n")
    f.write("=" * 70 + "\n")
    f.write(np.array2string(
        array,
        separator=", ",
        threshold=np.inf,
        max_line_width=200
    ))
    f.write("\n")


def show_summary(name, array):
    flat = array.reshape(-1)

    print(f"\n{name}")
    print(f"  Shape       : {array.shape}")
    print(f"  First 10    : {flat[:10]}")
    print(f"  Minimum     : {flat.min()}")
    print(f"  Maximum     : {flat.max()}")


# ============================================================
# MAIN
# ============================================================

samples = load_samples(SAMPLE_FILE)
weights = load_weights(WEIGHT_FILE)

if not 0 <= SAMPLE_INDEX < len(samples):
    raise ValueError(
        f"SAMPLE_INDEX must be between 0 and {len(samples)-1}"
    )

x = samples[SAMPLE_INDEX]

# ============================================================
# APPLY 8-BIT SIGNED HARDWARE CONSTRAINT
# ============================================================

def to_int8_hardware(x):
    """
    Mimics an 8-bit signed hardware bus.

    Range:
        -128 to +127

    Values outside this range wrap exactly as an 8-bit
    two's-complement hardware signal would.
    """
    return ((x + 128) % 256) - 128


x_raw = samples[SAMPLE_INDEX]

# Raw values from int_x.txt
x = to_int8_hardware(x_raw)

print("\nFirst two rows BEFORE 8-bit conversion:")
print(x_raw[:2])

print("\nFirst two rows AFTER 8-bit signed hardware conversion:")
print(x[:2])

print("\nFirst two rows of selected sample:")
print(x[:2])

# Get weights and biases.
w1, b1 = weights["conv1d_8"]
w2, b2 = weights["conv1d_9"]
w3, b3 = weights["dense_8"]
w4, b4 = weights["dense_9"]

print("\n========================================")
print("SOFTWARE NEURAL NETWORK REFERENCE")
print("========================================")

print(f"Number of samples : {len(samples)}")
print(f"Selected sample   : {SAMPLE_INDEX}")
print(f"Input shape       : {x.shape}")

print("\nWeight shapes:")
print("conv1d_8 :", w1.shape, " bias:", b1.shape)
print("conv1d_9 :", w2.shape, " bias:", b2.shape)
print("dense_8  :", w3.shape, " bias:", b3.shape)
print("dense_9  :", w4.shape, " bias:", b4.shape)


# ------------------------------------------------------------
# LAYER 1
# ------------------------------------------------------------

conv1 = conv1d(x, w1, b1)
relu1 = np.maximum(conv1, 0)

show_summary("Conv1 output", conv1)
show_summary("ReLU1 output", relu1)


# ------------------------------------------------------------
# MAX POOLING
# ------------------------------------------------------------

# Pool size = 2, stride = 2.
# 1018 becomes 509.

pool1 = relu1.reshape(
    relu1.shape[0] // 2,
    2,
    relu1.shape[1]
).max(axis=1)

show_summary("MaxPool output", pool1)


# ------------------------------------------------------------
# LAYER 2
# ------------------------------------------------------------

conv2 = conv1d(pool1, w2, b2)
relu2 = np.maximum(conv2, 0)

show_summary("Conv2 output", conv2)
show_summary("ReLU2 output", relu2)


# ------------------------------------------------------------
# GLOBAL AVERAGE POOLING
# ------------------------------------------------------------

gap = np.mean(relu2, axis=0)

show_summary("Global Average Pool output", gap)


# ------------------------------------------------------------
# DENSE 1
# ------------------------------------------------------------

dense1 = gap @ w3 + b3


show_summary("Dense1 output", dense1)



# ------------------------------------------------------------
# DENSE 2 / CLASSIFIER
# ------------------------------------------------------------

logits = relu @ w4 + b4

# Softmax is only used to get probabilities.
# Prediction itself is simply argmax(logits).
exp_logits = np.exp(logits - np.max(logits))
probabilities = exp_logits / np.sum(exp_logits)

prediction = int(np.argmax(logits))

show_summary("Final logits", logits)

print("\nFinal logits:")
print(logits)

print("\nSoftmax probabilities:")
print(probabilities)

print("\nPredicted class:", prediction)


# ============================================================
# SAVE ALL INTERMEDIATE VALUES
# ============================================================

with open(OUTPUT_FILE, "w") as f:

    f.write("SOFTWARE INTERMEDIATE VALUES\n")
    f.write(f"Sample index: {SAMPLE_INDEX}\n")

    write_array(f, "INPUT", x)

    write_array(f, "CONV1 OUTPUT", conv1)
    write_array(f, "RELU1 OUTPUT", relu1)

    write_array(f, "MAXPOOL OUTPUT", pool1)

    write_array(f, "CONV2 OUTPUT", conv2)
    write_array(f, "RELU2 OUTPUT", relu2)

    write_array(f, "GLOBAL AVERAGE POOL OUTPUT", gap)

    write_array(f, "DENSE1 OUTPUT", dense1)
    write_array(f, "RELU AFTER DENSE1", relu3)

    write_array(f, "FINAL LOGITS", logits)
    write_array(f, "SOFTMAX PROBABILITIES", probabilities)

    f.write("\n")
    f.write("=" * 70 + "\n")
    f.write(f"PREDICTED CLASS: {prediction}\n")
    f.write("=" * 70 + "\n")

print(f"\nFull intermediate values saved to:")
print(OUTPUT_FILE)


First two rows BEFORE 8-bit conversion:
[[ 35 -85  66  -8 -46  -5]
 [ 33 -82  65  -7 -42  -5]]

First two rows AFTER 8-bit signed hardware conversion:
[[ 35 -85  66  -8 -46  -5]
 [ 33 -82  65  -7 -42  -5]]

First two rows of selected sample:
[[ 35 -85  66  -8 -46  -5]
 [ 33 -82  65  -7 -42  -5]]

SOFTWARE NEURAL NETWORK REFERENCE
Number of samples : 150
Selected sample   : 36
Input shape       : (1019, 6)

Weight shapes:
conv1d_8 : (2, 6, 32)  bias: (32,)
conv1d_9 : (2, 32, 64)  bias: (64,)
dense_8  : (64, 64)  bias: (64,)
dense_9  : (64, 6)  bias: (6,)

Conv1 output
  Shape       : (1018, 32)
  First 10    : [ 15334 -13160  -3072  -7535  -6680  -5823   6370  -1996   3854 -15991]
  Minimum     : -43988
  Maximum     : 44538

ReLU1 output
  Shape       : (1018, 32)
  First 10    : [15334     0     0     0     0     0  6370     0  3854     0]
  Minimum     : 0
  Maximum     : 44538

MaxPool output
  Shape       : (509, 32)
  First 10    : [15334     0     0     0     0     0  6370     0

In [ ]:
#First multiplication and resulting sum of the convolution layer 1
print("Input position 0:", x[0])
print("Input position 1:", x[1])
print("Weights for output channel 0:")
print(w1[:, :, 0])
print("Bias:", b1[0])

terms = []
for k in range(2):
    for c in range(6):
        terms.append(
            (k, c, x[k,c], w1[k,c,0], x[k,c] * w1[k,c,0])
        )

for t in terms:
    print(t)

print("Sum:", sum(t[4] for t in terms) + b1[0])

Input position 0: [ 35 -85  66  -8 -46  -5]
Input position 1: [ 33 -82  65  -7 -42  -5]
Weights for output channel 0:
[[ 58 -20 -27 -39 -36  44]
 [ 56 -80   8  39 -64 -59]]
Bias: 0
(0, 0, np.int64(35), np.int64(58), np.int64(2030))
(0, 1, np.int64(-85), np.int64(-20), np.int64(1700))
(0, 2, np.int64(66), np.int64(-27), np.int64(-1782))
(0, 3, np.int64(-8), np.int64(-39), np.int64(312))
(0, 4, np.int64(-46), np.int64(-36), np.int64(1656))
(0, 5, np.int64(-5), np.int64(44), np.int64(-220))
(1, 0, np.int64(33), np.int64(56), np.int64(1848))
(1, 1, np.int64(-82), np.int64(-80), np.int64(6560))
(1, 2, np.int64(65), np.int64(8), np.int64(520))
(1, 3, np.int64(-7), np.int64(39), np.int64(-273))
(1, 4, np.int64(-42), np.int64(-64), np.int64(2688))
(1, 5, np.int64(-5), np.int64(-59), np.int64(295))
Sum: 15334


In [ ]:
# ============================================================
# DETAILED CONV2 MULTIPLICATION TRACE
# ============================================================

# Select which Conv2 output to inspect.
# Conv2 output shape = (508, 64)
#
# t = time position
# o = output channel

CONV2_TRACE_T = 0
CONV2_TRACE_CHANNEL = 0

print("\n")
print("=" * 80)
print("DETAILED CONV2 MULTIPLICATION TRACE")
print("=" * 80)

t = CONV2_TRACE_T
o = CONV2_TRACE_CHANNEL

print(f"Conv2 output position : {t}")
print(f"Conv2 output channel  : {o}")

print("\nInput window:")
print(f"pool1[{t}]   = {pool1[t]}")
print(f"pool1[{t+1}] = {pool1[t+1]}")

print("\nWeights for output channel:")
print(f"w2[0, :, {o}] = {w2[0, :, o]}")
print(f"w2[1, :, {o}] = {w2[1, :, o]}")

print(f"\nBias:")
print(f"b2[{o}] = {b2[o]}")

print("\n" + "-" * 80)
print("INDIVIDUAL MULTIPLICATIONS")
print("-" * 80)

total = 0

for k in range(2):
    print(f"\nKernel position k = {k}")

    for c in range(32):
        input_value = pool1[t + k, c]
        weight_value = w2[k, c, o]
        product = input_value * weight_value

        total += product

        print(
            f"  pool1[{t+k},{c}] = {input_value:8d}   "
            f"x   w2[{k},{c},{o}] = {weight_value:8d}"
            f"   ->   {product:12d}"
        )

print("\n" + "-" * 80)
print("ACCUMULATION")
print("-" * 80)

print(f"Sum of 64 products = {total}")
print(f"Bias               = {b2[o]}")
print(f"Conv2 result       = {total} + ({b2[o]})")
print(f"Conv2[{t},{o}]      = {total + b2[o]}")

print("\nPython Conv2 value:")
print(f"conv2[{t},{o}]      = {conv2[t, o]}")

print("\nReLU result:")
print(f"relu2[{t},{o}]      = {relu2[t, o]}")

if total + b2[o] == conv2[t, o]:
    print("\n✓ MANUAL CALCULATION MATCHES CONV2 OUTPUT")
else:
    print("\n✗ MISMATCH")



DETAILED CONV2 MULTIPLICATION TRACE
Conv2 output position : 0
Conv2 output channel  : 0

Input window:
pool1[0]   = [15334     0     0     0     0     0  6370     0  3868     0  2393     0
  6361  2350  6870     0     0   526     0     0     0     0     0     0
 18587  4916   443  4042     0  4827   969  4866]
pool1[1] = [13498     0     0     0     0     0  6054     0  3661     0  2372     0
  6187  1574  6559     0     0   818     0     0     0     0     0     0
 16978  4377   623  3421     0  4457   687  4167]

Weights for output channel:
w2[0, :, 0] = [ -4 -15  39  14  45 -23  -2  36  -7 -19  46 -35  47   3 -35  -5 -23 -38
 -13  -9 -11  -1  36 -23  23   9  41  -3  36  37 -17  -1]
w2[1, :, 0] = [-31  38  -3   9  27  45 -31  -1  42  -4  16 -25 -25  19  20  14  35 -30
  35  35 -22 -34 -40   9 -28   4   3  -8 -40 -39  30   7]

Bias:
b2[0] = 7

--------------------------------------------------------------------------------
INDIVIDUAL MULTIPLICATIONS
----------------------------------

In [ ]:
# ============================================================
# DETAILED CONV2 TRACE - NEXT STRIDE
# ============================================================

CONV2_TRACE_T = 1          # NEXT STRIDE
CONV2_TRACE_CHANNEL = 0    # Output channel to inspect

t = CONV2_TRACE_T
o = CONV2_TRACE_CHANNEL

print("\n")
print("=" * 80)
print("DETAILED CONV2 MULTIPLICATION TRACE")
print("=" * 80)

print(f"Conv2 output position : {t}")
print(f"Conv2 output channel  : {o}")

print("\nInput window:")
print(f"pool1[{t}]   = {pool1[t]}")
print(f"pool1[{t+1}] = {pool1[t+1]}")

print("\nWeights:")
print(f"w2[0, :, {o}] = {w2[0, :, o]}")
print(f"w2[1, :, {o}] = {w2[1, :, o]}")

print("\nBias:")
print(f"b2[{o}] = {b2[o]}")

print("\n" + "-" * 80)
print("INDIVIDUAL MULTIPLICATIONS")
print("-" * 80)

total = 0

for k in range(2):
    print(f"\nKernel position k = {k}")

    for c in range(32):

        input_value = pool1[t + k, c]
        weight_value = w2[k, c, o]
        product = input_value * weight_value

        total += product

        print(
            f"pool1[{t+k},{c}] = {input_value:8d} "
            f"x w2[{k},{c},{o}] = {weight_value:8d} "
            f"-> {product:12d}"
        )

print("\n" + "-" * 80)
print("FINAL ACCUMULATION")
print("-" * 80)

print(f"Sum of 64 products = {total}")
print(f"Bias               = {b2[o]}")
print(f"Conv2 result       = {total} + ({b2[o]})")
print(f"Conv2[{t},{o}]      = {total + b2[o]}")

print("\nPython reference:")
print(f"conv2[{t},{o}]      = {conv2[t, o]}")

print("\nReLU:")
print(f"relu2[{t},{o}]      = {relu2[t, o]}")

if total + b2[o] == conv2[t, o]:
    print("\n✓ MATCH")
else:
    print("\n✗ MISMATCH")



DETAILED CONV2 MULTIPLICATION TRACE
Conv2 output position : 1
Conv2 output channel  : 0

Input window:
pool1[1]   = [13498     0     0     0     0     0  6054     0  3661     0  2372     0
  6187  1574  6559     0     0   818     0     0     0     0     0     0
 16978  4377   623  3421     0  4457   687  4167]
pool1[2] = [11691     0     0     0     0     0  5977     0  3060     0  2327     0
  6187   832  6210     0     0  1124     0     0     0     0     0     0
 15317  3761   934  2741     0  4113   640  3586]

Weights:
w2[0, :, 0] = [ -4 -15  39  14  45 -23  -2  36  -7 -19  46 -35  47   3 -35  -5 -23 -38
 -13  -9 -11  -1  36 -23  23   9  41  -3  36  37 -17  -1]
w2[1, :, 0] = [-31  38  -3   9  27  45 -31  -1  42  -4  16 -25 -25  19  20  14  35 -30
  35  35 -22 -34 -40   9 -28   4   3  -8 -40 -39  30   7]

Bias:
b2[0] = 7

--------------------------------------------------------------------------------
INDIVIDUAL MULTIPLICATIONS
-----------------------------------------------------

In [ ]:
running_sum = 0

for i in range(relu2.shape[0]):
    value = relu2[i, 0]
    running_sum += value
    print(f"Position {i}: value = {value}, running sum = {running_sum}")

print("\nFinal sum =", running_sum)

Position 0: value = 0, running sum = 0
Position 1: value = 0, running sum = 0
Position 2: value = 0, running sum = 0
Position 3: value = 0, running sum = 0
Position 4: value = 0, running sum = 0
Position 5: value = 46555, running sum = 46555
Position 6: value = 407910, running sum = 454465
Position 7: value = 697814, running sum = 1152279
Position 8: value = 1318225, running sum = 2470504
Position 9: value = 1780246, running sum = 4250750
Position 10: value = 2269070, running sum = 6519820
Position 11: value = 2772508, running sum = 9292328
Position 12: value = 3194477, running sum = 12486805
Position 13: value = 3649991, running sum = 16136796
Position 14: value = 4128700, running sum = 20265496
Position 15: value = 4674500, running sum = 24939996
Position 16: value = 5351656, running sum = 30291652
Position 17: value = 6062368, running sum = 36354020
Position 18: value = 6786447, running sum = 43140467
Position 19: value = 7263763, running sum = 50404230
Position 20: value = 7530081,

In [ ]:
# ============================================
# Conv2 detailed calculation
# 11th time instant (t = 10)
# Output channel 0
# ============================================

t = 10
          # 11th time instant
oc = 0          # output channel

total = 0

print(f"Conv2[{t}, {oc}] calculation")
print("="*80)

for k in range(2):          # kernel size = 2
    for c in range(32):     # 32 input channels

        x_val = pool1[t + k, c]
        w_val = w2[k, c, oc]
        prod = x_val * w_val

        total += prod

        print(
            f"k={k:1d}, c={c:2d} : "
            f"{x_val:10d} * {w_val:6d} = {prod:12d}"
        )

print("="*80)
print("Bias =", b2[oc])
print("Sum before bias =", total)
print("Final output =", total + b2[oc])
print("Reference from conv2 =", conv2[t, oc])

Conv2[10, 0] calculation
k=0, c= 0 :       2459 *     -4 =        -9836
k=0, c= 1 :       6890 *    -15 =      -103350
k=0, c= 2 :       4598 *     39 =       179322
k=0, c= 3 :      11905 *     14 =       166670
k=0, c= 4 :      15419 *     45 =       693855
k=0, c= 5 :       7651 *    -23 =      -175973
k=0, c= 6 :          0 *     -2 =            0
k=0, c= 7 :          0 *     36 =            0
k=0, c= 8 :      11254 *     -7 =       -78778
k=0, c= 9 :          0 *    -19 =            0
k=0, c=10 :      11202 *     46 =       515292
k=0, c=11 :      15300 *    -35 =      -535500
k=0, c=12 :          0 *     47 =            0
k=0, c=13 :      17677 *      3 =        53031
k=0, c=14 :       7627 *    -35 =      -266945
k=0, c=15 :          0 *     -5 =            0
k=0, c=16 :          0 *    -23 =            0
k=0, c=17 :      13438 *    -38 =      -510644
k=0, c=18 :      14006 *    -13 =      -182078
k=0, c=19 :          0 *     -9 =            0
k=0, c=20 :          0 *    -11 =  

In [ ]:
# ============================================================
# STEP-BY-STEP DENSE1 CALCULATION
# ============================================================

# Dense1:
#   gap      = (64,)
#   w3       = (64, 64)
#   b3       = (64,)
#
# Dense1[j] = sum(gap[i] * w3[i,j]) + b3[j]

print("\n" + "=" * 80)
print("DENSE1 STEP-BY-STEP CALCULATION")
print("=" * 80)

# Choose which Dense1 neuron/output you want to inspect.
# 0 means the first Dense1 output.
OUTPUT_NEURON = 0

print(f"\nCalculating Dense1 output neuron [{OUTPUT_NEURON}]")
print("-" * 80)

running_sum = 0

for i in range(64):

    gap_value = gap[i]
    weight_value = w3[i, OUTPUT_NEURON]

    product = gap_value * weight_value
    running_sum += product

    print(
        f"GAP[{i:2d}] = {gap_value:15.6f}   "
        f"x   W[{i:2d},{OUTPUT_NEURON:2d}] = {weight_value:6d}   "
        f"= {product:18.6f}   "
        f"Running Sum = {running_sum:18.6f}"
    )

print("-" * 80)

print(f"Sum before bias = {running_sum:.6f}")
print(f"Bias[{OUTPUT_NEURON}] = {b3[OUTPUT_NEURON]}")
print(
    f"Dense1[{OUTPUT_NEURON}] = "
    f"{running_sum + b3[OUTPUT_NEURON]:.6f}"
)

print(f"Python Dense1[{OUTPUT_NEURON}] = {dense1[OUTPUT_NEURON]:.6f}")

# Verification
if np.isclose(
    running_sum + b3[OUTPUT_NEURON],
    dense1[OUTPUT_NEURON]
):
    print("\n✓ Calculation matches Dense1 output.")
else:
    print("\n✗ Calculation DOES NOT match Dense1 output.")


DENSE1 STEP-BY-STEP CALCULATION

Calculating Dense1 output neuron [0]
--------------------------------------------------------------------------------
GAP[ 0] =   286371.222441   x   W[ 0, 0] =     28   =     8018394.228346   Running Sum =     8018394.228346
GAP[ 1] =    18247.627953   x   W[ 1, 0] =    -18   =     -328457.303150   Running Sum =     7689936.925197
GAP[ 2] =   394432.704724   x   W[ 2, 0] =     -9   =    -3549894.342520   Running Sum =     4140042.582677
GAP[ 3] =    80192.316929   x   W[ 3, 0] =     37   =     2967115.726378   Running Sum =     7107158.309055
GAP[ 4] =   478023.145669   x   W[ 4, 0] =     55   =    26291273.011811   Running Sum =    33398431.320866
GAP[ 5] =   445751.446850   x   W[ 5, 0] =     -8   =    -3566011.574803   Running Sum =    29832419.746063
GAP[ 6] =    11819.295276   x   W[ 6, 0] =    -13   =     -153650.838583   Running Sum =    29678768.907480
GAP[ 7] =     2331.072835   x   W[ 7, 0] =     37   =       86249.694882   Running Sum =    

In [ ]:
# ============================================================
# HARDWARE-ACCURATE 34-BIT SIGNED ADDITION
# ============================================================

def signed_34(value):
    """
    Mimics a signed [33:0] Verilog signal.

    Range:
        -2^33 to 2^33 - 1
    """

    value = int(value)

    # Keep only lower 34 bits
    value = value & ((1 << 34) - 1)

    # Convert to signed two's complement
    if value & (1 << 33):
        value -= (1 << 34)

    return value

In [ ]:
# ============================================================
# DENSE 2 - STEP BY STEP CALCULATION
# NO QUANTISATION / NO 34-BIT WRAPPING
# NO ReLU3
# ============================================================

# Dense 2 input = dense1 directly
dense2_input = dense1

# Dense 2 weights and biases
dense2_weights = w4
dense2_bias = b4

print("\n" + "=" * 80)
print("DENSE 2 - STEP BY STEP CALCULATION")
print("NO QUANTISATION / NO 34-BIT WRAPPING")
print("=" * 80)

print("Input shape :", dense2_input.shape)
print("Weight shape:", dense2_weights.shape)
print("Bias shape  :", dense2_bias.shape)


# ============================================================
# DENSE 2 - ADDER TREE
# 64 MULTIPLICATIONS
# ============================================================

dense2_output = np.zeros(6, dtype=np.float64)

for out_neuron in range(6):

    print("\n" + "=" * 80)
    print(f"DENSE2 OUTPUT NEURON {out_neuron}")
    print("=" * 80)

    # --------------------------------------------------------
    # 64 PARALLEL MULTIPLICATIONS
    # --------------------------------------------------------

    products = np.zeros(64, dtype=np.float64)

    for i in range(64):

        products[i] = (
            float(dense2_input[i]) *
            float(dense2_weights[i, out_neuron])
        )

    # --------------------------------------------------------
    # LEVEL 1
    # 64 -> 32
    # --------------------------------------------------------

    q = np.zeros(32, dtype=np.float64)

    for i in range(32):
        q[i] = products[2*i] + products[2*i + 1]

    # --------------------------------------------------------
    # LEVEL 2
    # 32 -> 16
    # --------------------------------------------------------

    w = np.zeros(16, dtype=np.float64)

    for i in range(16):
        w[i] = q[2*i] + q[2*i + 1]

    # --------------------------------------------------------
    # LEVEL 3
    # 16 -> 8
    # --------------------------------------------------------

    e = np.zeros(8, dtype=np.float64)

    for i in range(8):
        e[i] = w[2*i] + w[2*i + 1]

    # --------------------------------------------------------
    # LEVEL 4
    # 8 -> 4
    # --------------------------------------------------------

    r = np.zeros(4, dtype=np.float64)

    for i in range(4):
        r[i] = e[2*i] + e[2*i + 1]

    # --------------------------------------------------------
    # LEVEL 5
    # 4 -> 2
    # --------------------------------------------------------

    t = np.zeros(2, dtype=np.float64)

    for i in range(2):
        t[i] = r[2*i] + r[2*i + 1]

    # --------------------------------------------------------
    # FINAL ADDITION + BIAS
    # --------------------------------------------------------

    bias = float(dense2_bias[out_neuron])

    final_output = t[0] + t[1] + bias

    dense2_output[out_neuron] = final_output

    # ========================================================
    # PRINT RESULTS
    # ========================================================

    print("\n64 MULTIPLICATIONS:")
    for i in range(64):
        print(
            f"Product[{i+1:2d}] = "
            f"{dense2_input[i]} × "
            f"{dense2_weights[i, out_neuron]} "
            f"= {products[i]}"
        )

    print("\n32 ADDITIONS:")
    for i in range(32):
        print(
            f"q[{i}] = Product[{2*i+1}] + "
            f"Product[{2*i+2}] = {q[i]}"
        )

    print("\n16 ADDITIONS:")
    for i in range(16):
        print(
            f"w[{i}] = q[{2*i}] + "
            f"q[{2*i+1}] = {w[i]}"
        )

    print("\n8 ADDITIONS:")
    for i in range(8):
        print(
            f"e[{i}] = w[{2*i}] + "
            f"w[{2*i+1}] = {e[i]}"
        )

    print("\n4 ADDITIONS:")
    for i in range(4):
        print(
            f"r[{i}] = e[{2*i}] + "
            f"e[{2*i+1}] = {r[i]}"
        )

    print("\n2 ADDITIONS:")
    for i in range(2):
        print(
            f"t[{i}] = r[{2*i}] + "
            f"r[{2*i+1}] = {t[i]}"
        )

    print("\nFINAL:")
    print(f"t[0] = {t[0]}")
    print(f"t[1] = {t[1]}")
    print(f"bias  = {bias}")

    print(
        f"Dense2 output[{out_neuron}] = "
        f"{t[0]} + {t[1]} + ({bias})"
    )

    print(
        f"Dense2 output[{out_neuron}] = "
        f"{final_output}"
    )


# ============================================================
# FINAL DENSE2 OUTPUT
# ============================================================

print("\n" + "=" * 80)
print("FINAL DENSE 2 OUTPUTS")
print("=" * 80)

print(dense2_output)


# ============================================================
# DIRECT MATRIX MULTIPLICATION CHECK
# ============================================================

print("\n" + "=" * 80)
print("DIRECT MATRIX MULTIPLICATION CHECK")
print("=" * 80)

dense2_check = (
    dense2_input @ dense2_weights.astype(np.float64)
    + dense2_bias.astype(np.float64)
)

print(dense2_check)


DENSE 2 - STEP BY STEP CALCULATION
NO QUANTISATION / NO 34-BIT WRAPPING
Input shape : (64,)
Weight shape: (64, 6)
Bias shape  : (6,)

DENSE2 OUTPUT NEURON 0

64 MULTIPLICATIONS:
Product[ 1] = -32074860.637795277 × -43 = 1379219007.425197
Product[ 2] = -75372864.3996063 × -3 = 226118593.19881892
Product[ 3] = 53009314.03346458 × 62 = 3286577470.074804
Product[ 4] = 61671268.614173226 × 7 = 431698880.2992126
Product[ 5] = 86931912.07283463 × 62 = 5389778548.515747
Product[ 6] = 22481673.488188986 × -53 = -1191528694.8740163
Product[ 7] = -64510041.05708663 × -86 = 5547863530.909451
Product[ 8] = 108649604.97834647 × 42 = 4563283409.090551
Product[ 9] = -62714727.338582665 × 8 = -501717818.7086613
Product[10] = 52915444.72637797 × 8 = 423323557.8110238
Product[11] = -35768716.71062991 × -63 = 2253429152.7696843
Product[12] = 51235684.34055118 × 69 = 3535262219.4980316
Product[13] = 26363124.377952754 × -3 = -79089373.13385826
Product[14] = 110010036.0 × 47 = 5170471692.0
Product[15] = -4